# Heaps & Priority Queues: Zero to Hero

**NB-09 in the [DSA: Zero to Hero](README.md) series.**

A tree with a **weaker** invariant than NB-07's, stored in a **flat array** with no pointers at
all — which is how it answers the locality complaint NB-04 §3.1 and NB-08 have been making about
pointer-based structures for four notebooks.

***

## Why this notebook is different

- **The $O(n)$ heapify claim is proved, then measured, then found to be more interesting than the
  claim.** Bottom-up heapify does **at most 1.00 swaps per element** on any input — 0.74 on random,
  0.00 on ascending, 1.00 on descending. Building by repeated `push` is *also* linear on random
  input (1.27/element) and only becomes $\Theta(n\log n)$ on adversarial input, where it grows to
  **18 swaps per element and rising**. So "heapify is $O(n)$, push-building is $O(n\log n)$" is
  half right, and the half that is true is the half about **who chooses the input**.
- **The cache argument for d-ary heaps is measured, and it disagrees with the comparison count.**
  A 4-ary heap does *more* comparisons than a binary heap (31.6 against 30.3 per element) and is
  **faster in wall clock** (1.68 s against 2.12 s), because it has half as many levels and each
  node's children are contiguous.
- **The decrease-key question gets a genuinely mixed answer.** Like-for-like in Python, an indexed
  heap beats lazy deletion (5.35 s against 7.54 s) — the algorithmic argument is real. But the C
  library's heap with lazy deletion beats both (2.41 s). Which is why Java's `PriorityQueue` has no
  decrease-key and nobody misses it.

And the signature difficulty, which is the most common misconception in this notebook's subject:
**a heap is not sorted, and its array is not a sorted array.** §3 demonstrates it directly.

***

## Contents

**Part 1 — Theory from zero**
1. The heap property, and the implicit array layout
2. Sift-up, sift-down, push and pop
3. **Heapify in $O(n)$** — the proof, and what the measurement adds to it
4. **d-ary heaps** and the cache argument
5. Java: `PriorityQueue`, and the missing decrease-key

**Part 2 — Worked problems** — top-k, the running median, k-way merge, heapsort, indexed heaps
**Part 3 — The signature difficulty: a heap is not sorted**
**Part 4 — Tough questions** · **Part 5 — Practice** · **Part 6 — Reading**

***

## In one paragraph

A **binary heap** is a complete binary tree satisfying one weak invariant — **every node is $\le$
both its children** (for a min-heap) — which says nothing about the relationship between siblings
and therefore imposes only a *partial* order. That weakness is what makes it cheap: the minimum is
at the root, available in $O(1)$, while insert and extract cost $O(\log n)$ and nothing else is
ordered at all. Because the tree is always **complete**, it needs no pointers: node $i$'s children
live at $2i+1$ and $2i+2$ and its parent at $\lfloor (i-1)/2 \rfloor$, so the entire structure is a
**flat array** — which is why a heap has the locality that NB-04 §3.1 measured pointer-based trees
losing by a factor of 30. Two operations maintain the invariant: **sift-up** after inserting at the
end, and **sift-down** after moving the last element to the root. Building a heap from $n$ existing
elements can be done in **$\Theta(n)$**, not $\Theta(n \log n)$, by sifting down from the last
internal node — an argument worth seeing because the summation is genuinely surprising. The
resulting **priority queue** is the right structure whenever you repeatedly need the extreme of a
changing collection: top-k, k-way merge, heapsort, event simulation, Huffman coding, and Dijkstra's
algorithm (NB-21). What it is bad at is everything else — search, arbitrary deletion, and ordered
iteration are all $\Theta(n)$, because the invariant never promised them.

**Prerequisites:** [NB-06 Trees](trees_zero_to_hero.ipynb) for tree vocabulary and the complete-tree
shape, [NB-08 Balanced Trees](balanced_trees_zero_to_hero.ipynb) for the contrast — this is the
other way to guarantee $O(\log n)$, by weakening the invariant instead of working to maintain a
strong one — and [NB-01 Arrays](arrays_zero_to_hero.ipynb) §3 for the locality argument §1.1 cashes
in.

***
# Part 0 - Setup

Standard library only, plus `dsa_toolkit`. §1.5 needs the JDK.

Python's `heapq` is used as the **reference implementation** throughout — every heap this notebook
builds is differentially tested against it.

In [1]:
# ---------------------------------------------------------------------------
# Everything this notebook uses. Standard library only.
# ---------------------------------------------------------------------------
import heapq
import math
import random
import statistics
import sys
import time

from dsa_toolkit import (InvariantError, JavaError, StressFailure, check_invariant,
                         cross_check, growth_table, java_available, measure_growth,
                         run_java, stress)

RANDOM_SEED = 12345

ok, detail = java_available()
JAVA = ok
print("python", sys.version.split()[0])
print("JDK available:", ok, "|", detail)

python 3.14.7
JDK available: True | javac 25.0.4.1


***
# Part 1 - Theory from zero

1. The heap property, and the implicit array layout
2. Sift-up, sift-down, push and pop
3. **Heapify in $O(n)$** — the proof, and what the measurement adds
4. **d-ary heaps** and the cache argument
5. Java: `PriorityQueue`, and the missing decrease-key

## 1.1 The heap property, and the implicit array layout

**The invariant** (min-heap):

> Every node's key is $\le$ the keys of both its children.

Compare that with NB-07's BST invariant, which related a node to *every* key in each subtree. This
one relates a node only to its two children, and says **nothing at all** about siblings. It is a
**partial order**, and the weakness is the entire design:

- the **minimum is at the root**, because it is $\le$ its children, which are $\le$ theirs, and so
  on — the only thing the invariant guarantees;
- **nothing else is located.** The second smallest is one of the root's two children, but the
  seventh smallest could be almost anywhere. §3 is about this.

**The second half of the definition is a shape constraint:** a heap is a **complete** binary tree —
every level full except possibly the last, which fills left to right. That has a consequence worth
stating loudly:

> **A complete tree needs no pointers.** Number the nodes in level order from 0 and the
> relationships are arithmetic:
> $$\text{left}(i) = 2i+1, \qquad \text{right}(i) = 2i+2, \qquad \text{parent}(i) = \left\lfloor \frac{i-1}{2} \right\rfloor$$

So a heap of $n$ elements is **an array of $n$ elements**. No node objects, no left/right fields, no
allocation per element — which is NB-04 §1.1's 48-bytes-per-node and NB-04 §3.1's 30× traversal
penalty both going to zero. The tree is *implicit*: it exists only in how you index the array.

This is the trick worth taking away beyond heaps. **Whenever a tree is complete, it can live in an
array**, and the pointer-chasing complaint disappears. It works here precisely because the heap
property is weak enough to be maintainable while keeping the tree complete — a BST cannot do this,
because maintaining *its* invariant requires shapes that are not complete.

In [2]:
# ---------------------------------------------------------------------------
# 1.1 The implicit layout, made concrete.
# ---------------------------------------------------------------------------
def children(i):
    return 2 * i + 1, 2 * i + 2


def parent(i):
    return (i - 1) // 2


heap = [1, 3, 6, 5, 9, 8]
print("array:", heap)
print()
print("           1                 index 0")
print("         /   \\")
print("        3     6              indices 1, 2")
print("       / \\   /")
print("      5   9 8               indices 3, 4, 5")
print()
print("  %5s %8s %14s %14s" % ("index", "value", "children", "parent"))
print("  " + "-" * 46)
for i, value in enumerate(heap):
    left, right = children(i)
    kids = [heap[c] for c in (left, right) if c < len(heap)]
    print("  %5d %8d %14s %14s"
          % (i, value, kids if kids else "-", heap[parent(i)] if i else "-"))

print()
print("Every relationship is arithmetic on the index. There are no pointers in")
print("this structure at all -- which is NB-04's 48 bytes per node and its 30x")
print("traversal penalty both going to zero.")
print()
print("And note what the invariant does NOT say: 3 and 6 are siblings and the")
print("heap has no opinion about their order. Only parent-child pairs are")
print("constrained, which is why the array is not sorted (Part 3).")

array: [1, 3, 6, 5, 9, 8]

           1                 index 0
         /   \
        3     6              indices 1, 2
       / \   /
      5   9 8               indices 3, 4, 5

  index    value       children         parent
  ----------------------------------------------
      0        1         [3, 6]              -
      1        3         [5, 9]              1
      2        6            [8]              1
      3        5              -              3
      4        9              -              3
      5        8              -              6

Every relationship is arithmetic on the index. There are no pointers in
this structure at all -- which is NB-04's 48 bytes per node and its 30x
traversal penalty both going to zero.

And note what the invariant does NOT say: 3 and 6 are siblings and the
heap has no opinion about their order. Only parent-child pairs are
constrained, which is why the array is not sorted (Part 3).


## 1.2 Sift-up, sift-down, push and pop

Two repair operations, each restoring the invariant after it is broken in one specific place.

**`push`** appends to the end of the array — which keeps the tree complete — and then **sifts up**:
while the new element is smaller than its parent, swap. It stops when the parent is smaller, and it
travels at most the height of the tree, so $O(\log n)$.

**`pop`** must remove the root. It cannot just delete it — that would leave a hole at the top — so
it moves the **last** element to the root (keeping the tree complete) and **sifts down**: while the
element is larger than its smallest child, swap with that child. Also $O(\log n)$.

**Why sift-down compares against the *smaller* child** is the detail that trips people up: swapping
with the larger child would put the larger one above the smaller one, breaking the invariant on the
other side. You have to promote the smaller.

`peek` is $O(1)$ — the root is `a[0]`.

Everything below is differentially tested against `heapq`, with the invariant checked after every
operation.

In [3]:
# ---------------------------------------------------------------------------
# 1.2 A binary min-heap in a flat array.
# ---------------------------------------------------------------------------
SWAPS = {"up": 0, "down": 0}


class MinHeap:
    """Node i has children 2i+1 and 2i+2. No pointers anywhere."""

    def __init__(self, items=None, build="heapify"):
        self._a = []
        if items:
            if build == "heapify":
                self._a = list(items)
                self._heapify()
            else:
                for x in items:
                    self.push(x)

    def __len__(self):
        return len(self._a)

    def peek(self):
        if not self._a:
            raise IndexError("peek at an empty heap")
        return self._a[0]                          # O(1): the invariant puts it here

    def _sift_up(self, i):
        a = self._a
        while i > 0:
            p = (i - 1) // 2
            if a[p] <= a[i]:
                break
            a[p], a[i] = a[i], a[p]
            SWAPS["up"] += 1
            i = p

    def _sift_down(self, i):
        a, n = self._a, len(self._a)
        while True:
            smallest, left, right = i, 2 * i + 1, 2 * i + 2
            if left < n and a[left] < a[smallest]:
                smallest = left
            if right < n and a[right] < a[smallest]:
                smallest = right                   # the SMALLER child, or the swap
            if smallest == i:                      # would break the other side
                return
            a[i], a[smallest] = a[smallest], a[i]
            SWAPS["down"] += 1
            i = smallest

    def _heapify(self):
        """Bottom-up: leaves are already heaps, so start at the last internal node."""
        for i in range(len(self._a) // 2 - 1, -1, -1):
            self._sift_down(i)

    def push(self, x):
        self._a.append(x)                          # keeps the tree complete
        self._sift_up(len(self._a) - 1)

    def pop(self):
        a = self._a
        if not a:
            raise IndexError("pop from an empty heap")
        top = a[0]
        last = a.pop()                             # keeps the tree complete
        if a:
            a[0] = last
            self._sift_down(0)
        return top

    def as_list(self):
        return list(self._a)


def heap_ok(h):
    """The invariant: every node is <= both its children."""
    a = h._a
    for i in range(1, len(a)):
        p = (i - 1) // 2
        if a[p] > a[i]:
            return "heap property broken: a[%d]=%r > a[%d]=%r" % (p, a[p], i, a[i])
    return True


def gen_ops(rng):
    return [(rng.choice(["push", "push", "pop"]), rng.randrange(-30, 30))
            for _ in range(rng.randrange(0, 60))]


def replay(ops):
    heap, ref = MinHeap(), []
    popped = []
    for op, x in ops:
        if op == "push":
            heap.push(x)
            heapq.heappush(ref, x)
        else:
            if ref:
                got, want = heap.pop(), heapq.heappop(ref)
                assert got == want, "pop gave %r, expected %r" % (got, want)
                popped.append(got)
            else:
                try:
                    heap.pop()
                    raise AssertionError("pop on an empty heap should raise")
                except IndexError:
                    pass
        check_invariant(heap, heap_ok, "heap property", "%s %r" % (op, x))
        assert len(heap) == len(ref)
        if ref:
            assert heap.peek() == ref[0], "peek disagreed"
    return popped


def reference(ops):
    ref, popped = [], []
    for op, x in ops:
        if op == "push":
            heapq.heappush(ref, x)
        elif ref:
            popped.append(heapq.heappop(ref))
    return popped


checked = stress(replay, reference, gen_ops, n=5000, seed=RANDOM_SEED, label="MinHeap")
print("MinHeap: %s randomised push/pop sequences match heapq exactly," % "{:,}".format(checked))
print("         with the heap property checked after every single operation.")

heap = MinHeap()
for x in [5, 3, 8, 1, 9, 2]:
    heap.push(x)
print()
print("  pushed 5, 3, 8, 1, 9, 2")
print("  array now:", heap.as_list(), " peek:", heap.peek())
print("  popping everything:", [heap.pop() for _ in range(6)])

MinHeap: 5,000 randomised push/pop sequences match heapq exactly,
         with the heap property checked after every single operation.

  pushed 5, 3, 8, 1, 9, 2
  array now: [1, 3, 2, 5, 9, 8]  peek: 1
  popping everything: [1, 2, 3, 5, 8, 9]


## 1.3 Heapify in $O(n)$ — the proof, and what the measurement adds

Given $n$ elements already in an array, make them a heap. The obvious way is $n$ pushes, which
looks like $\Theta(n\log n)$. There is a better way, and the argument for it is genuinely
surprising.

**The algorithm.** Sift **down** from the last internal node backwards to the root. Every leaf is
already a valid heap of one element, so the second half of the array needs no work at all; and
processing backwards means that when you sift down at node $i$, both its subtrees are already
heaps.

**Why it is $\Theta(n)$, not $\Theta(n\log n)$.** The cost of sifting down is proportional to the
node's **height**, and *most nodes are near the bottom, where the height is small*:

| height $h$ | nodes at that height | work each | total |
|---|---|---|---|
| 0 (leaves) | $n/2$ | 0 | 0 |
| 1 | $n/4$ | 1 | $n/4$ |
| 2 | $n/8$ | 2 | $2n/8$ |
| $h$ | $\le n/2^{h+1}$ | $h$ | $hn/2^{h+1}$ |

Summing: $\sum_{h\ge 0} \frac{n\,h}{2^{h+1}} = \frac{n}{2}\sum_{h\ge 0}\frac{h}{2^{h}} = \frac{n}{2}\cdot 2 = n$

**So the total is at most $n$ swaps.** The whole argument turns on $\sum h/2^h = 2$ — a convergent
series, which is why the $\log n$ factor disappears. The $n$ pushes version is worse precisely
because it works the other way round: it sifts *up* from the bottom, and most nodes are at the
bottom, so most nodes travel the full height.

Now the measurement, which turns out to say something the claim does not.

In [4]:
# ---------------------------------------------------------------------------
# 1.3 Swaps to build a heap: heapify vs repeated push, on three input orders.
# ---------------------------------------------------------------------------
def build_swaps(values, method):
    SWAPS["up"] = SWAPS["down"] = 0
    MinHeap(values, build=method)
    return SWAPS["up"] + SWAPS["down"]


def make_input(kind, n):
    if kind == "ascending":
        return list(range(n))
    if kind == "descending":
        return list(range(n, 0, -1))
    rng = random.Random(n)                    # ONE generator, reused -- see the note below
    return [rng.randrange(1 << 30) for _ in range(n)]


checked = stress(lambda xs: heap_ok(MinHeap(xs, build="heapify")), lambda xs: True,
                 lambda r: [r.randrange(-40, 40) for _ in range(r.randrange(0, 40))],
                 n=4000, seed=RANDOM_SEED, label="heapify property")
print("heapify: %s random arrays satisfy the heap property afterwards."
      % "{:,}".format(checked))
checked = stress(lambda xs: sorted(MinHeap(xs, build="heapify").as_list()),
                 lambda xs: sorted(xs),
                 lambda r: [r.randrange(-40, 40) for _ in range(r.randrange(0, 40))],
                 n=4000, seed=RANDOM_SEED, label="heapify contents")
print("         %s of them still hold exactly the same multiset."
      % "{:,}".format(checked))

print()
print("Swaps to build a heap of n elements. The proof says heapify is at most n.")
print()
print("  %-12s %10s %14s %12s %16s %14s"
      % ("input", "n", "heapify", "per elem", "repeated push", "per elem"))
print("  " + "-" * 82)
for kind in ("ascending", "random", "descending"):
    for n in (10_000, 100_000, 1_000_000):
        values = make_input(kind, n)
        hf, pu = build_swaps(values, "heapify"), build_swaps(values, "push")
        print("  %-12s %10s %14s %12.2f %16s %14.2f"
              % (kind if n == 10_000 else "", "{:,}".format(n),
                 "{:,}".format(hf), hf / n, "{:,}".format(pu), pu / n))
    print()

heapify: 4,000 random arrays satisfy the heap property afterwards.
         4,000 of them still hold exactly the same multiset.

Swaps to build a heap of n elements. The proof says heapify is at most n.

  input                 n        heapify     per elem    repeated push       per elem
  ----------------------------------------------------------------------------------
  ascending        10,000              0         0.00                0           0.00


                  100,000              0         0.00                0           0.00


                1,000,000              0         0.00                0           0.00

  random           10,000          7,403         0.74           12,689           1.27
                  100,000         74,237         0.74          127,472           1.27


                1,000,000        744,276         0.74        1,282,059           1.28

  descending       10,000          9,992         1.00          113,631          11.36


                  100,000         99,990         1.00        1,468,946          14.69


                1,000,000        999,988         1.00       17,951,445          17.95



**Heapify never exceeds 1.00 swaps per element**, on any of the three inputs, at any size. The
proof said "at most $n$"; the measurement says the constant is between 0 and 1 and does not drift.

**But look at the `repeated push` column, because it does not say what the textbook claim implies.**

- On **ascending** input, push does **zero** swaps — every new element is the largest so far, so it
  stays where it lands. Heapify also does zero. Both are $\Theta(n)$.
- On **random** input, push does about **1.27 swaps per element** — a *constant*, so it is $\Theta(n)$
  too, merely 1.7× worse than heapify. Not $\Theta(n\log n)$ at all.
- On **descending** input, push does **11.4, then 14.7, then 18.0** swaps per element as $n$ grows
  by factors of ten. That column is the only one that grows, and it is growing like $\log n$: the
  ratio to $n\log_2 n$ converges towards 0.9.

So the familiar claim — *heapify is $O(n)$ and building by pushes is $O(n\log n)$* — is **only true
for adversarial input**. On random data both are linear and the difference is a constant factor of
1.7.

The honest statement is the one this series keeps arriving at: **heapify's cost is bounded at $n$
regardless of input; repeated push's cost is a property of the data**, ranging from $0$ to
$\Theta(n \log n)$. That is NB-05 §3.3's monotonic stack, NB-07 §3's BST and NB-02 §3's string
matching, for the fourth time — and it is why the bounded algorithm is the one you can make a
promise about.

**A note on how this cell generates its random input**, because getting it wrong invalidates the
whole table. `make_input` creates **one** generator and reuses it. Writing
`[random.Random(n).randrange(...) for _ in range(n)]` re-seeds on every element and produces $n$
identical values — which makes every column read 0.00 and looks like a triumph for both methods.
That is NB-00 §3.1's bug, and it happened again while this notebook was being written.

## 1.4 d-ary heaps and the cache argument

Nothing forces two children. A **d-ary heap** gives each node $d$ children at indices
$di+1 \ldots di+d$, with parent $\lfloor (i-1)/d \rfloor$. The tree becomes shallower —
$\log_d n$ levels instead of $\log_2 n$ — and the trade is visible immediately:

- **`push` gets cheaper**: sift-up compares against one parent per level, and there are fewer
  levels. $O(\log_d n)$.
- **`pop` gets more expensive**: sift-down must find the *smallest of $d$ children* at each level,
  so $d-1$ comparisons per level over $\log_d n$ levels — $O(d \log_d n)$, which as a function of
  $d$ is minimised near $d = e \approx 3$.

That is the comparison-counting analysis, and it predicts $d=2$ and $d=4$ should be about equal
with everything larger being worse. **The measurement disagrees with the comparison count in an
instructive way**, and the reason is the thing this whole notebook is about: the array layout.

In [5]:
# ---------------------------------------------------------------------------
# 1.4 d-ary heaps: comparisons predicted, and wall clock measured.
# ---------------------------------------------------------------------------
class DaryHeap:
    """A d-ary min-heap. Node i has children d*i+1 .. d*i+d."""

    def __init__(self, d=2, items=()):
        self.d = d
        self.comparisons = 0
        self._a = list(items)
        # The last INTERNAL node is at (n-2)//d. Writing n//d - 1 happens to be
        # right for d = 2 and is wrong for every other d -- caught by the
        # differential test below at d = 3, not by reading the code.
        for i in range((len(self._a) - 2) // d, -1, -1):
            self._sift_down(i)

    def __len__(self):
        return len(self._a)

    def _sift_down(self, i):
        a, n, d = self._a, len(self._a), self.d
        while True:
            first = d * i + 1
            if first >= n:
                return
            best = first
            for c in range(first + 1, min(first + d, n)):
                self.comparisons += 1
                if a[c] < a[best]:
                    best = c
            self.comparisons += 1
            if a[best] >= a[i]:
                return
            a[i], a[best] = a[best], a[i]
            i = best

    def _sift_up(self, i):
        a, d = self._a, self.d
        while i > 0:
            p = (i - 1) // d
            self.comparisons += 1
            if a[p] <= a[i]:
                return
            a[p], a[i] = a[i], a[p]
            i = p

    def push(self, x):
        self._a.append(x)
        self._sift_up(len(self._a) - 1)

    def pop(self):
        a = self._a
        top = a[0]
        last = a.pop()
        if a:
            a[0] = last
            self._sift_down(0)
        return top


def drain(d, values):
    """Build ONE heap, then pop it dry. (Building inside the loop would pop the
    same minimum every time -- a comprehension is not a substitute for state.)"""
    heap = DaryHeap(d, values)
    return [heap.pop() for _ in range(len(values))]


for d in (2, 3, 4, 8):
    checked = stress(lambda xs, d=d: drain(d, xs), lambda xs: sorted(xs),
                     lambda r: [r.randrange(-50, 50) for _ in range(r.randrange(0, 40))],
                     n=2000, seed=RANDOM_SEED, label="DaryHeap d=%d" % d)
    print("DaryHeap d=%d: %s random arrays pop in fully sorted order."
          % (d, "{:,}".format(checked)))

N = 200_000
rng = random.Random(RANDOM_SEED)
values = [rng.randrange(1 << 30) for _ in range(N)]

print()
print("n = %s. Build by heapify, then pop everything." % "{:,}".format(N))
print()
print("  %5s %10s %18s %14s %16s %12s"
      % ("d", "levels", "comparisons", "per element", "d*log_d(n)", "seconds"))
print("  " + "-" * 82)
for d in (2, 4, 8, 16, 32):
    heap = DaryHeap(d, values)
    heap.comparisons = 0
    start = time.perf_counter()
    while len(heap):
        heap.pop()
    elapsed = time.perf_counter() - start
    print("  %5d %10.1f %18s %14.1f %16.1f %12.2f"
          % (d, math.log(N, d), "{:,}".format(heap.comparisons),
             heap.comparisons / N, d * math.log(N, d), elapsed))
    del heap

print()
print("The comparison count tracks d*log_d(n) closely, and it says d=2 and d=4")
print("are equivalent with everything larger being worse.")
print()
print("The stopwatch disagrees: d=4 is the fastest, doing MORE comparisons than")
print("d=2 in LESS time. The reason is the array layout. A 4-ary heap has half")
print("as many levels, so half as many jumps to a distant part of the array --")
print("and the four children it compares are ADJACENT, so they arrive together")
print("in one cache line. It trades comparisons, which are cheap, for cache")
print("misses, which are not (NB-01 section 3).")

DaryHeap d=2: 2,000 random arrays pop in fully sorted order.


DaryHeap d=3: 2,000 random arrays pop in fully sorted order.
DaryHeap d=4: 2,000 random arrays pop in fully sorted order.


DaryHeap d=8: 2,000 random arrays pop in fully sorted order.



n = 200,000. Build by heapify, then pop everything.

      d     levels        comparisons    per element       d*log_d(n)      seconds
  ----------------------------------------------------------------------------------


      2       17.6          6,062,622           30.3             35.2         2.04


      4        8.8          6,324,506           31.6             35.2         1.52


      8        5.9          8,689,797           43.4             47.0         1.67


     16        4.4         13,437,858           67.2             70.4         2.04


     32        3.5         22,444,666          112.2            112.7         2.68

The comparison count tracks d*log_d(n) closely, and it says d=2 and d=4
are equivalent with everything larger being worse.

The stopwatch disagrees: d=4 is the fastest, doing MORE comparisons than
d=2 in LESS time. The reason is the array layout. A 4-ary heap has half
as many levels, so half as many jumps to a distant part of the array --
and the four children it compares are ADJACENT, so they arrive together
in one cache line. It trades comparisons, which are cheap, for cache
misses, which are not (NB-01 section 3).


## 1.5 Java: `PriorityQueue`, and the missing decrease-key

`java.util.PriorityQueue` is a binary min-heap in an array — the same structure as §1.2 — ordered
by `Comparable` or a supplied `Comparator` (NB-07 §1.4's contract applies here too).

**The thing it does not have is `decrease_key`**, and neither does Python's `heapq`, and the reason
is structural: to lower an element's priority you must first *find* it, and a heap has no index.
Searching is $\Theta(n)$ (§3), so `decreaseKey` would be $\Theta(n)$ — which is worse than useless
for the algorithm that wants it most, Dijkstra's (NB-21).

**Two ways out**, and §2.5 measures both:

1. **An indexed heap** — maintain a `key → array position` map alongside the heap, updated on every
   swap. Then `decrease_key` is $O(\log n)$. This is what the textbook Dijkstra analysis assumes.
2. **Lazy deletion** — do not update anything. Push a *new* entry with the better priority, leave
   the stale one in the heap, and discard stale entries when they surface. Simpler, uses more
   memory, and is what essentially everybody actually does.

Note also `PriorityQueue.remove(Object)` **exists** and is $O(n)$, because it does the linear
search §3 describes. It compiles, it looks like a normal collection operation, and it is the
easiest way to make a heap-based algorithm quadratic.

In [6]:
# ---------------------------------------------------------------------------
# 1.5 What PriorityQueue offers, and what it costs.
# ---------------------------------------------------------------------------
JAVA_PQ_SRC = r"""
import java.util.*;

public class PQ {
    static double best(Runnable r, int reps) {
        double b = Double.MAX_VALUE;
        for (int i = 0; i < reps; i++) {
            long t0 = System.nanoTime();
            r.run();
            b = Math.min(b, (System.nanoTime() - t0) / 1e6);
        }
        return b;
    }

    public static void main(String[] args) {
        System.out.println("A. PriorityQueue is a binary heap -- its iterator is NOT sorted:");
        PriorityQueue<Integer> pq = new PriorityQueue<>(List.of(5, 3, 8, 1, 9, 2));
        System.out.println("   built from [5, 3, 8, 1, 9, 2]");
        System.out.println("   toString()      = " + pq + "   <- heap order, not sorted");
        System.out.println("   peek()          = " + pq.peek() + "   <- the only ordered guarantee");
        List<Integer> drained = new ArrayList<>();
        PriorityQueue<Integer> copy = new PriorityQueue<>(pq);
        while (!copy.isEmpty()) drained.add(copy.poll());
        System.out.println("   polling it dry  = " + drained + "   <- sorted, but only this way");

        System.out.println();
        System.out.println("B. There is no decreaseKey. remove(Object) exists and is O(n):");
        final int n = 200_000;
        final PriorityQueue<Integer> big = new PriorityQueue<>();
        for (int i = 0; i < n; i++) big.add(i);
        final Random rnd = new Random(1);
        final int[] targets = new int[2_000];
        for (int i = 0; i < targets.length; i++) targets[i] = rnd.nextInt(n);

        double polls = best(() -> {
            PriorityQueue<Integer> q = new PriorityQueue<>(big);
            for (int i = 0; i < 2_000; i++) q.poll();
        }, 3);
        double removes = best(() -> {
            PriorityQueue<Integer> q = new PriorityQueue<>(big);
            for (int t : targets) q.remove(t);
        }, 3);
        System.out.printf("   2,000 poll()   on %,d elements: %8.1f ms   (O(log n) each)%n", n, polls);
        System.out.printf("   2,000 remove() on %,d elements: %8.1f ms   (O(n) each)%n", n, removes);
        System.out.printf("   remove is %.0fx slower, and it is one method call away.%n",
                          removes / polls);

        System.out.println();
        System.out.println("C. Heapify is O(n): building from a collection beats n adds.");
        final List<Integer> data = new ArrayList<>();
        for (int i = n; i > 0; i--) data.add(i);          // descending: push's worst case
        double byAdds = best(() -> {
            PriorityQueue<Integer> q = new PriorityQueue<>();
            for (int x : data) q.add(x);
        }, 5);
        double byHeapify = best(() -> { PriorityQueue<Integer> q = new PriorityQueue<>(data); }, 5);
        System.out.printf("   %,d add() calls           : %7.1f ms%n", n, byAdds);
        System.out.printf("   new PriorityQueue(list)   : %7.1f ms   (%.1fx faster)%n",
                          byHeapify, byAdds / byHeapify);
    }
}
"""

if JAVA:
    print(run_java(JAVA_PQ_SRC, timeout=900))
else:
    print("JDK not available; skipping the Java section.")

A. PriorityQueue is a binary heap -- its iterator is NOT sorted:
   built from [5, 3, 8, 1, 9, 2]
   toString()      = [1, 3, 2, 5, 9, 8]   <- heap order, not sorted
   peek()          = 1   <- the only ordered guarantee
   polling it dry  = [1, 2, 3, 5, 8, 9]   <- sorted, but only this way

B. There is no decreaseKey. remove(Object) exists and is O(n):
   2,000 poll()   on 200,000 elements:      1.8 ms   (O(log n) each)
   2,000 remove() on 200,000 elements:    146.5 ms   (O(n) each)
   remove is 80x slower, and it is one method call away.

C. Heapify is O(n): building from a collection beats n adds.
   200,000 add() calls           :     9.8 ms
   new PriorityQueue(list)   :     2.2 ms   (4.4x faster)



Three things, and the middle one is the trap.

**A** is Part 3's subject arriving early: `PriorityQueue.toString()` prints the **array**, which is
heap order, not sorted order. Only `peek` is guaranteed, and the sorted sequence exists only if you
drain the queue.

**B** is the operation you should never call. `remove(Object)` is a linear scan followed by a
sift, so it is $\Theta(n)$ — and it is spelled exactly like the $O(\log n)$ `poll`. Calling it in a
loop is the standard way to turn an $O(n \log n)$ algorithm into an $O(n^2)$ one, and nothing in the
type system objects.

**C** confirms §1.3 in a different language and runtime: constructing a `PriorityQueue` from a
collection uses heapify and beats $n$ separate `add` calls on push's worst-case input.

**The decrease-key question is genuinely open**, which is why §2.5 measures it rather than
asserting the textbook answer.

***
# Part 2 - Worked problems

| # | Problem | The idea |
|---|---|---|
| 2.1 | Top-k | a heap of size **k**, not **n** |
| 2.2 | Running median | **two** heaps facing each other |
| 2.3 | k-way merge | one heap holding one element per source |
| 2.4 | Heapsort | the heap used backwards, in place |
| 2.5 | Indexed heaps vs lazy deletion | the decrease-key question, measured |

## 2.1 Top-k — a heap of size k

**The problem.** The $k$ largest of $n$ items, with $k \ll n$.

**The move that matters:** keep a **min**-heap of size $k$ — a min-heap, to find the *largest*
elements. The root is then the *smallest of the current best k*, which is exactly the element to
evict when something better arrives. Push each item; if the heap exceeds $k$, pop.

$\Theta(n \log k)$ time and **$\Theta(k)$ space**, against $\Theta(n \log n)$ and $\Theta(n)$ for
sorting everything. When $k$ is 10 and $n$ is a billion, that space bound is the whole point: this
works on a **stream** you cannot store.

The wrong instinct is a max-heap of all $n$ items popped $k$ times. That is $\Theta(n + k\log n)$ —
often fine, and it needs all $n$ in memory, which is the thing you were trying to avoid.

In [7]:
# ---------------------------------------------------------------------------
# 2.1 Top-k with a size-k min-heap.
# ---------------------------------------------------------------------------
def top_k(stream, k):
    """The k largest, in descending order. O(n log k) time, O(k) space."""
    if k <= 0:
        return []
    heap = []                                   # min-heap of the best k so far
    for x in stream:
        if len(heap) < k:
            heapq.heappush(heap, x)
        elif x > heap[0]:                       # better than the worst we keep
            heapq.heapreplace(heap, x)          # pop and push in one sift
    return sorted(heap, reverse=True)


def gen_topk(rng):
    n = rng.randrange(0, 40)
    return ([rng.randrange(-50, 50) for _ in range(n)], rng.randrange(0, 12))


checked = stress(lambda c: top_k(c[0], c[1]),
                 lambda c: sorted(c[0], reverse=True)[:c[1]],
                 gen_topk, n=4000, seed=RANDOM_SEED, label="top_k")
print("top_k: %s random (stream, k) pairs agree with sorting and slicing,"
      % "{:,}".format(checked))
print("       including k = 0, k > len(stream) and duplicate values.")

print()
N, K = 2_000_000, 10
rng = random.Random(RANDOM_SEED)
data = [rng.randrange(1 << 30) for _ in range(N)]

start = time.perf_counter()
by_heap = top_k(data, K)
t_heap = time.perf_counter() - start

start = time.perf_counter()
by_sort = sorted(data, reverse=True)[:K]
t_sort = time.perf_counter() - start

assert by_heap == by_sort
print("n = %s, k = %d:" % ("{:,}".format(N), K))
print("  size-k heap : %.3f s, holding %d elements" % (t_heap, K))
print("  full sort   : %.3f s, holding %s elements" % (t_sort, "{:,}".format(N)))
print()
print("  The sort is competitive on time -- Timsort is very fast and this data")
print("  is in memory already. The heap's win is the SPACE: it never holds more")
print("  than 10 items, so it works on a stream that does not fit in memory,")
print("  and the sort simply cannot run at all in that setting.")
del data, by_sort

top_k: 4,000 random (stream, k) pairs agree with sorting and slicing,
       including k = 0, k > len(stream) and duplicate values.



n = 2,000,000, k = 10:
  size-k heap : 0.121 s, holding 10 elements
  full sort   : 0.662 s, holding 2,000,000 elements

  The sort is competitive on time -- Timsort is very fast and this data
  is in memory already. The heap's win is the SPACE: it never holds more
  than 10 items, so it works on a stream that does not fit in memory,
  and the sort simply cannot run at all in that setting.


## 2.2 Running median — two heaps facing each other

**The problem.** Numbers arrive one at a time; report the median after each.

**The idea.** Split the values in half and keep each half in a heap **facing the middle**:

- a **max**-heap for the lower half, so its root is the *largest of the small values*;
- a **min**-heap for the upper half, so its root is the *smallest of the large values*.

The two roots are the two elements adjacent to the median, so the median is available in $O(1)$.
Each insertion is $O(\log n)$: push into the appropriate side, then **rebalance** so the sizes
differ by at most one.

**The invariant to hold onto**, and to assert:

> every element of the low heap $\le$ every element of the high heap, and
> $0 \le |{\rm low}| - |{\rm high}| \le 1$.

**Python has no max-heap**, so the low half stores **negated** values in a min-heap. That is the
standard trick and the standard source of sign bugs, which is why the invariant checks the actual
ordering rather than trusting it.

In [8]:
# ---------------------------------------------------------------------------
# 2.2 A running median from two heaps.
# ---------------------------------------------------------------------------
class RunningMedian:
    """low is a max-heap (negated); high is a min-heap. low may hold one extra."""

    def __init__(self):
        self._low = []                          # negated values
        self._high = []

    def __len__(self):
        return len(self._low) + len(self._high)

    def add(self, x):
        if self._low and x <= -self._low[0]:
            heapq.heappush(self._low, -x)
        else:
            heapq.heappush(self._high, x)
        # rebalance: low holds ceil(n/2), high holds floor(n/2)
        if len(self._low) > len(self._high) + 1:
            heapq.heappush(self._high, -heapq.heappop(self._low))
        elif len(self._high) > len(self._low):
            heapq.heappush(self._low, -heapq.heappop(self._high))

    def median(self):
        if not self._low:
            raise IndexError("median of nothing")
        if len(self._low) > len(self._high):
            return float(-self._low[0])
        return (-self._low[0] + self._high[0]) / 2.0


def median_ok(m):
    if m._low and m._high and -m._low[0] > m._high[0]:
        return "low root %r exceeds high root %r" % (-m._low[0], m._high[0])
    gap = len(m._low) - len(m._high)
    if gap not in (0, 1):
        return "sizes %d and %d differ by %d" % (len(m._low), len(m._high), gap)
    return True


def running_medians(values):
    m, out = RunningMedian(), []
    for x in values:
        m.add(x)
        check_invariant(m, median_ok, "two-heap invariant", "add %r" % x)
        out.append(m.median())
    return out


def median_reference(values):
    out, seen = [], []
    for x in values:
        bisect.insort(seen, x)
        n = len(seen)
        out.append(float(seen[n // 2]) if n % 2 else (seen[n // 2 - 1] + seen[n // 2]) / 2.0)
    return out


import bisect

checked = stress(running_medians, median_reference,
                 lambda r: [r.randrange(-40, 40) for _ in range(r.randrange(0, 40))],
                 n=4000, seed=RANDOM_SEED, label="RunningMedian")
print("RunningMedian: %s random streams match a sorted-list reference at EVERY"
      % "{:,}".format(checked))
print("               step, with the two-heap invariant checked after each add.")

print()
demo = [5, 15, 1, 3, 8, 7, 9, 10, 6, 11]
m = RunningMedian()
print("  %8s %10s %28s" % ("value", "median", "low (max-heap) | high (min-heap)"))
print("  " + "-" * 54)
for x in demo:
    m.add(x)
    print("  %8d %10.1f %14s | %-14s"
          % (x, m.median(), sorted((-v for v in m._low), reverse=True), sorted(m._high)))

RunningMedian: 4,000 random streams match a sorted-list reference at EVERY
               step, with the two-heap invariant checked after each add.

     value     median low (max-heap) | high (min-heap)
  ------------------------------------------------------
         5        5.0            [5] | []            
        15       10.0            [5] | [15]          
         1        5.0         [5, 1] | [15]          
         3        4.0         [3, 1] | [5, 15]       
         8        5.0      [5, 3, 1] | [8, 15]       
         7        6.0      [5, 3, 1] | [7, 8, 15]    
         9        7.0   [7, 5, 3, 1] | [8, 9, 15]    
        10        7.5   [7, 5, 3, 1] | [8, 9, 10, 15]
         6        7.0 [7, 6, 5, 3, 1] | [8, 9, 10, 15]
        11        7.5 [7, 6, 5, 3, 1] | [8, 9, 10, 11, 15]


## 2.3 k-way merge — one heap, one element per source

**The problem.** Merge $k$ sorted sequences into one.

**The idea.** A heap holding **one element from each source** — the current front. Pop the smallest,
emit it, and push the next element from *that* source. The heap never exceeds $k$ entries.

$\Theta(N \log k)$ for $N$ elements total, against $\Theta(N \log N)$ for concatenating and sorting
— and, again, $\Theta(k)$ space, so the sources can be files, network streams, or anything else too
large to hold.

**This is the merge step of external sorting**, which is how you sort more data than you have
memory: sort chunks that fit, write them out, then k-way merge the sorted runs. Python's
`heapq.merge` does exactly this.

The detail that makes it work is what you put *in* the heap: `(value, source_index, iterator)`.
The source index is needed to know where to refill from, and it also **breaks ties without
comparing the iterators**, which are not orderable — the same trap as NB-07's `TreeSet` comparator,
in a different costume.

In [9]:
# ---------------------------------------------------------------------------
# 2.3 Merging k sorted sequences with a heap of size k.
# ---------------------------------------------------------------------------
def k_way_merge(sequences):
    """Merge k sorted sequences. O(N log k) time, O(k) space."""
    heap = []
    iterators = [iter(s) for s in sequences]
    for i, it in enumerate(iterators):
        first = next(it, None)
        if first is not None:
            # the index breaks ties, so the iterator is never compared
            heapq.heappush(heap, (first, i))
    out = []
    while heap:
        value, i = heapq.heappop(heap)
        out.append(value)
        nxt = next(iterators[i], None)
        if nxt is not None:
            heapq.heappush(heap, (nxt, i))
    return out


def gen_sequences(rng):
    k = rng.randrange(0, 7)
    return [sorted(rng.randrange(-30, 30) for _ in range(rng.randrange(0, 12)))
            for _ in range(k)]


checked = stress(k_way_merge,
                 lambda seqs: sorted(x for s in seqs for x in s),
                 gen_sequences, n=4000, seed=RANDOM_SEED, label="k_way_merge")
print("k_way_merge: %s random families of sorted sequences merge correctly,"
      % "{:,}".format(checked))
print("             including empty families and empty sequences.")

print()
runs = [[1, 4, 9], [2, 3, 10], [5, 6, 7, 8]]
print("  merging", runs)
print("  ->", k_way_merge(runs))

print()
K, PER = 200, 2_000
rng = random.Random(RANDOM_SEED)
sources = [sorted(rng.randrange(1 << 30) for _ in range(PER)) for _ in range(K)]
total = K * PER

start = time.perf_counter()
merged = k_way_merge(sources)
t_merge = time.perf_counter() - start
start = time.perf_counter()
concatenated = sorted(x for s in sources for x in s)
t_sort = time.perf_counter() - start
assert merged == concatenated

print("%d sorted runs of %s elements (%s total):"
      % (K, "{:,}".format(PER), "{:,}".format(total)))
print("  k-way merge : %.3f s, heap never exceeds %d entries" % (t_merge, K))
print("  concat+sort : %.3f s, holds all %s at once" % (t_sort, "{:,}".format(total)))
print()
print("  Timsort wins on time again -- it detects the sorted runs and merges")
print("  them in C. The heap's argument is space and streaming, not speed:")
print("  this is how you sort data larger than memory.")
del sources, merged, concatenated

k_way_merge: 4,000 random families of sorted sequences merge correctly,
             including empty families and empty sequences.

  merging [[1, 4, 9], [2, 3, 10], [5, 6, 7, 8]]
  -> [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]



200 sorted runs of 2,000 elements (400,000 total):
  k-way merge : 0.255 s, heap never exceeds 200 entries
  concat+sort : 0.088 s, holds all 400,000 at once

  Timsort wins on time again -- it detects the sorted runs and merges
  them in C. The heap's argument is space and streaming, not speed:
  this is how you sort data larger than memory.


## 2.4 Heapsort — the heap used backwards, in place

**The algorithm.** Heapify the array into a **max**-heap ($\Theta(n)$), then repeatedly swap the
root with the last element and sift down over the shrinking prefix. After $n-1$ rounds the array is
sorted **ascending**, in place.

The trick is that the max-heap grows the sorted region from the *right*: each extracted maximum
goes exactly where it belongs, at the end of the unsorted prefix, so no extra array is needed.

| | Heapsort | Quicksort | Mergesort |
|---|---|---|---|
| worst case | **$\Theta(n\log n)$ guaranteed** | $\Theta(n^2)$ | $\Theta(n\log n)$ |
| extra space | **$\Theta(1)$** | $\Theta(\log n)$ stack | $\Theta(n)$ |
| stable | no | no | **yes** |
| in practice | **slowest of the three** | usually fastest | predictable |

**So why is the one with the best worst case and the best space bound the least used?** Locality.
Quicksort scans contiguous runs; heapsort's sift-down jumps to $2i+1$, then $4i+3$, then $8i+7$ —
strides that double every level and leave the cache almost immediately. It is the one place in this
notebook where the array layout is *not* enough, because the access pattern within the array is
still scattered.

Where it earns its place: **introsort**, the algorithm actually in `std::sort`, runs quicksort and
switches to heapsort when the recursion gets too deep — using it as a guaranteed-$O(n\log n)$
backstop against quicksort's quadratic case. NB-13 covers this properly.

In [10]:
# ---------------------------------------------------------------------------
# 2.4 Heapsort, in place.
# ---------------------------------------------------------------------------
def heapsort(values):
    """Sort ascending, in place, using a MAX-heap. Theta(n log n), Theta(1) space."""
    a = list(values)
    n = len(a)

    def sift_down(i, size):
        while True:
            largest, left, right = i, 2 * i + 1, 2 * i + 2
            if left < size and a[left] > a[largest]:
                largest = left
            if right < size and a[right] > a[largest]:
                largest = right
            if largest == i:
                return
            a[i], a[largest] = a[largest], a[i]
            i = largest

    for i in range((n - 2) // 2, -1, -1):        # build the max-heap: Theta(n)
        sift_down(i, n)
    for end in range(n - 1, 0, -1):              # extract, growing the sorted tail
        a[0], a[end] = a[end], a[0]
        sift_down(0, end)
    return a


checked = stress(heapsort, sorted,
                 lambda r: [r.randrange(-60, 60) for _ in range(r.randrange(0, 50))],
                 n=5000, seed=RANDOM_SEED, label="heapsort")
print("heapsort: %s random arrays sort identically to sorted()," % "{:,}".format(checked))
print("          including empty, single-element and all-equal inputs.")

print()
print("Growth on adversarial input -- heapsort has no bad case to construct:")
print()
for label, make in (("random", lambda n: [random.Random(n).randrange(1 << 30)] * 0
                     or [random.Random(n + 1).randrange(1 << 30) for _ in range(n)]),
                    ("already sorted", lambda n: list(range(n))),
                    ("reverse sorted", lambda n: list(range(n, 0, -1)))):
    rows = measure_growth(heapsort, [20_000, 40_000, 80_000], setup=make, repeats=3)
    ratios = " ".join("%.2f" % rows[i]["ratio"] for i in range(1, len(rows)))
    print("  %-16s ratios %s   (n log n predicts ~2.1)" % (label, ratios))

heapsort: 5,000 random arrays sort identically to sorted(),
          including empty, single-element and all-equal inputs.

Growth on adversarial input -- heapsort has no bad case to construct:



  random           ratios 2.01 2.25   (n log n predicts ~2.1)


  already sorted   ratios 2.12 2.18   (n log n predicts ~2.1)


  reverse sorted   ratios 2.12 2.17   (n log n predicts ~2.1)


## 2.5 Indexed heaps versus lazy deletion — the decrease-key question

§1.5 said neither `heapq` nor `PriorityQueue` offers `decrease_key`, and gave two workarounds. This
section measures them, because the textbook answer and the practical answer disagree.

**The workload** is Dijkstra's shape (NB-21): insert every vertex with distance $\infty$, then
lower priorities many times as shorter paths are found, then extract in priority order.

**The indexed heap** maintains a `key → position` map, updated on **every swap**, so an element can
be located in $O(1)$ and sifted up in $O(\log n)$. The map is the whole implementation, and keeping
it correct through every swap is exactly NB-08 §2.2's augmentation problem — so the invariant checks
it after every operation.

**Lazy deletion** does nothing clever: push a *new* entry with the better priority, leave the stale
one behind, and skip stale entries when they surface. The heap grows beyond $n$, which is the cost.

In [11]:
# ---------------------------------------------------------------------------
# 2.5 An indexed heap with decrease_key.
# ---------------------------------------------------------------------------
class IndexedMinHeap:
    """A min-heap that can find any key in O(1) via a position map."""

    def __init__(self):
        self._a = []                            # (priority, key)
        self._pos = {}                          # key -> index in _a

    def __len__(self):
        return len(self._a)

    def __contains__(self, key):
        return key in self._pos

    def _swap(self, i, j):
        a = self._a
        a[i], a[j] = a[j], a[i]
        self._pos[a[i][1]] = i                  # the map must follow EVERY swap
        self._pos[a[j][1]] = j

    def _sift_up(self, i):
        while i > 0:
            p = (i - 1) // 2
            if self._a[p] <= self._a[i]:
                return
            self._swap(p, i)
            i = p

    def _sift_down(self, i):
        a, n = self._a, len(self._a)
        while True:
            smallest, left, right = i, 2 * i + 1, 2 * i + 2
            if left < n and a[left] < a[smallest]:
                smallest = left
            if right < n and a[right] < a[smallest]:
                smallest = right
            if smallest == i:
                return
            self._swap(i, smallest)
            i = smallest

    def push(self, key, priority):
        if key in self._pos:
            raise KeyError("duplicate key %r" % (key,))
        self._a.append((priority, key))
        self._pos[key] = len(self._a) - 1
        self._sift_up(len(self._a) - 1)

    def decrease_key(self, key, priority):
        i = self._pos[key]
        if priority > self._a[i][0]:
            raise ValueError("decrease_key must not increase the priority")
        self._a[i] = (priority, key)
        self._sift_up(i)                        # a decrease can only move it UP

    def pop(self):
        a = self._a
        if not a:
            raise IndexError("pop from an empty heap")
        top = a[0]
        last = a.pop()
        del self._pos[top[1]]
        if a:
            a[0] = last
            self._pos[last[1]] = 0
            self._sift_down(0)
        return top[1], top[0]


def indexed_ok(heap):
    """Heap property PLUS the position map -- the augmentation must be right too."""
    a = heap._a
    if len(heap._pos) != len(a):
        return "position map holds %d entries, heap holds %d" % (len(heap._pos), len(a))
    for i, (_, key) in enumerate(a):
        if heap._pos.get(key) != i:
            return "pos[%r] is %r but the element sits at %d" % (key, heap._pos.get(key), i)
    for i in range(1, len(a)):
        if a[(i - 1) // 2] > a[i]:
            return "heap property broken at index %d" % i
    return True


def gen_indexed(rng):
    keys = list(range(rng.randrange(1, 15)))
    ops = [("push", k, rng.randrange(100)) for k in keys]
    ops += [(rng.choice(["decrease", "pop"]), rng.choice(keys), rng.randrange(100))
            for _ in range(rng.randrange(0, 25))]
    return ops


def replay_indexed(ops):
    heap, ref = IndexedMinHeap(), {}
    out = []
    for op, key, priority in ops:
        if op == "push":
            if key not in ref:
                heap.push(key, priority)
                ref[key] = priority
        elif op == "decrease":
            if key in ref and priority <= ref[key]:
                heap.decrease_key(key, priority)
                ref[key] = priority
        else:
            if ref:
                got_key, got_priority = heap.pop()
                want = min(ref.items(), key=lambda kv: (kv[1], kv[0]))
                assert (got_priority, got_key) == (want[1], want[0]), \
                    "popped %r, expected %r" % ((got_priority, got_key), want)
                del ref[got_key]
                out.append((got_key, got_priority))
        check_invariant(heap, indexed_ok, "indexed heap", "%s %r" % (op, key))
    return out


def indexed_reference(ops):
    ref, out = {}, []
    for op, key, priority in ops:
        if op == "push":
            if key not in ref:
                ref[key] = priority
        elif op == "decrease":
            if key in ref and priority <= ref[key]:
                ref[key] = priority
        else:
            if ref:
                want = min(ref.items(), key=lambda kv: (kv[1], kv[0]))
                del ref[want[0]]
                out.append(want)
    return out


checked = stress(replay_indexed, indexed_reference, gen_indexed,
                 n=4000, seed=RANDOM_SEED, label="IndexedMinHeap")
print("IndexedMinHeap: %s randomised sequences agree with a dictionary reference,"
      % "{:,}".format(checked))
print("                with the POSITION MAP verified after every operation --")
print("                the same augmentation discipline as NB-08 section 2.2.")

IndexedMinHeap: 4,000 randomised sequences agree with a dictionary reference,
                with the POSITION MAP verified after every operation --
                the same augmentation discipline as NB-08 section 2.2.


In [12]:
# ---------------------------------------------------------------------------
# And the comparison the textbooks and practitioners disagree about.
# ---------------------------------------------------------------------------
class PlainHeap:
    """The same pure-Python binary heap, WITHOUT the position map -- so the
    comparison below is like-for-like rather than Python against C."""

    def __init__(self):
        self._a = []

    def __len__(self):
        return len(self._a)

    def push(self, x):
        a = self._a
        a.append(x)
        i = len(a) - 1
        while i > 0:
            p = (i - 1) // 2
            if a[p] <= a[i]:
                break
            a[p], a[i] = a[i], a[p]
            i = p

    def pop(self):
        a = self._a
        top = a[0]
        last = a.pop()
        if a:
            a[0] = last
            i, n = 0, len(a)
            while True:
                s, left, right = i, 2 * i + 1, 2 * i + 2
                if left < n and a[left] < a[s]:
                    s = left
                if right < n and a[right] < a[s]:
                    s = right
                if s == i:
                    break
                a[i], a[s] = a[s], a[i]
                i = s
        return top


INF = 1 << 21


def run_indexed(n, decreases):
    heap = IndexedMinHeap()
    for k in range(n):
        heap.push(k, INF)
    peak = len(heap)
    for k, p in decreases:
        if k in heap and p < heap._a[heap._pos[k]][0]:
            heap.decrease_key(k, p)
    while len(heap):
        heap.pop()
    return peak


def run_lazy(n, decreases, heap_factory, push, pop, empty):
    heap = heap_factory()
    best = {}
    for k in range(n):
        best[k] = INF
        push(heap, (INF, k))
    peak = n
    for k, p in decreases:
        if p < best[k]:
            best[k] = p
            push(heap, (p, k))                  # the stale entry stays behind
            peak = max(peak, len(heap))
    done = set()
    while not empty(heap):
        p, k = pop(heap)
        if k in done or p != best[k]:           # stale: skip it
            continue
        done.add(k)
    return peak


print("Dijkstra-shaped workload: n pushes, 3n priority decreases, then drain.")
print()
print("  %10s %16s %20s %18s %16s"
      % ("n", "indexed (Py)", "lazy PlainHeap (Py)", "lazy heapq (C)", "peak heap size"))
print("  " + "-" * 86)
for n in (50_000, 200_000):
    rng = random.Random(RANDOM_SEED)
    decreases = [(rng.randrange(n), rng.randrange(1 << 20)) for _ in range(n * 3)]

    start = time.perf_counter()
    indexed_peak = run_indexed(n, decreases)
    t_indexed = time.perf_counter() - start

    start = time.perf_counter()
    lazy_peak = run_lazy(n, decreases, PlainHeap, lambda h, x: h.push(x),
                         lambda h: h.pop(), lambda h: len(h) == 0)
    t_plain = time.perf_counter() - start

    start = time.perf_counter()
    run_lazy(n, decreases, list, heapq.heappush, heapq.heappop, lambda h: not h)
    t_heapq = time.perf_counter() - start

    print("  %10s %16.2f %20.2f %18.2f %16s"
          % ("{:,}".format(n), t_indexed, t_plain, t_heapq,
             "%s vs %s" % ("{:,}".format(indexed_peak), "{:,}".format(lazy_peak))))

Dijkstra-shaped workload: n pushes, 3n priority decreases, then drain.

           n     indexed (Py)  lazy PlainHeap (Py)     lazy heapq (C)   peak heap size
  --------------------------------------------------------------------------------------


      50,000             0.89                 1.24               0.30 50,000 vs 134,490


     200,000             4.75                 6.17               1.77 200,000 vs 537,732


**Both arguments turn out to be right, about different things.**

**Like-for-like in Python, the indexed heap wins** — it is meaningfully faster than lazy deletion
using the *same* hand-written heap, because the heap never grows past $n$ while the lazy version's
grows to roughly 2.7×. The textbook analysis is not wrong: eliminating stale entries really does
less work.

**And the C library's heap with lazy deletion beats both.** `heapq` is implemented in C, and the
constant-factor gap is larger than the algorithmic saving — the same lesson as NB-04 §3 and NB-08
§2.4, arriving again: *your clever structure competes with their fast simple one, and the fast
simple one usually wins.*

So the practical guidance:

- **Use lazy deletion with the library heap.** It is a dozen lines, has no position map to corrupt,
  and is what Java's `PriorityQueue`, most Dijkstra implementations, and every event-simulation loop
  actually do.
- **The cost is memory**, not time: the heap holds stale entries until they surface. Bound it by
  reheapifying if it exceeds some multiple of $n$, if that matters.
- **Reach for an indexed heap** when memory is genuinely tight, or when you need to *query* an
  element's current priority, which lazy deletion cannot answer without a separate map anyway.

**And be careful about what the asymptotic analysis says.** Dijkstra with a binary heap is
$O((V+E)\log V)$ either way — lazy deletion pushes at most $E$ entries, so the heap is $O(E)$ and
$\log E = O(\log V)$ for simple graphs. Lazy deletion does not change the complexity class. It only
looked like it should.

***
# Part 3 - The signature difficulty: a heap is not sorted

This notebook's difficulty is not an adversarial input or a subtle piece of code. It is a
**misconception**, and it is the most common one in the subject:

> **A heap is not sorted, and its array is not a sorted array.**

The confusion is understandable. A heap *produces* sorted output when you drain it, `heapsort`
sorts, the minimum is always at the front, and printing a small heap often *looks* sorted by
accident. But the invariant only relates **parents to children**. It says nothing about siblings,
nothing about cousins, and nothing about any two elements not on the same root-to-leaf path.

The consequences are the whole of "what heaps are bad at", and each one is a real bug people write.

In [13]:
# ---------------------------------------------------------------------------
# 3.1 The array is not sorted -- and how often it happens to look sorted.
# ---------------------------------------------------------------------------
rng = random.Random(RANDOM_SEED)
values = [rng.randrange(100) for _ in range(12)]
heap = MinHeap(values)

print("input   ", values)
print("heapified", heap.as_list())
print("sorted   ", sorted(values))
print()
print("  heap array == sorted?", heap.as_list() == sorted(values))
print("  is the heap property satisfied?", heap_ok(heap) is True)
print()
print("Both can be true at once: it IS a valid heap and it is NOT sorted.")

print()
print("How often does a heapified array happen to come out sorted?")
print()
print("  %8s %16s %14s" % ("n", "trials", "sorted by luck"))
print("  " + "-" * 44)
for n in (2, 3, 4, 6, 10):
    trials, accidents = 3000, 0
    r = random.Random(n)
    for _ in range(trials):
        xs = [r.randrange(1000) for _ in range(n)]
        if MinHeap(xs).as_list() == sorted(xs):
            accidents += 1
    print("  %8d %16s %13.1f%%" % (n, "{:,}".format(trials), 100.0 * accidents / trials))

print()
print("At n = 2 a heap is ALWAYS sorted -- there is only one parent-child pair,")
print("so the two constraints coincide. That is why toy examples mislead: the")
print("smallest cases are exactly the ones where the distinction vanishes.")

input    [53, 93, 1, 38, 47, 24, 34, 72, 55, 20, 47, 15]
heapified [1, 20, 15, 38, 47, 24, 34, 72, 55, 93, 47, 53]
sorted    [1, 15, 20, 24, 34, 38, 47, 47, 53, 55, 72, 93]

  heap array == sorted? False
  is the heap property satisfied? True

Both can be true at once: it IS a valid heap and it is NOT sorted.

How often does a heapified array happen to come out sorted?

         n           trials sorted by luck
  --------------------------------------------
         2            3,000         100.0%
         3            3,000          52.0%
         4            3,000          34.4%
         6            3,000           5.4%
        10            3,000           0.0%

At n = 2 a heap is ALWAYS sorted -- there is only one parent-child pair,
so the two constraints coincide. That is why toy examples mislead: the
smallest cases are exactly the ones where the distinction vanishes.


In [14]:
# ---------------------------------------------------------------------------
# 3.2 What follows: the four things a heap cannot do.
# ---------------------------------------------------------------------------
big = MinHeap([rng.randrange(1 << 20) for _ in range(200_000)])
array = big.as_list()

print("A heap of %s elements." % "{:,}".format(len(array)))
print()

print("1. SEARCH is Theta(n), not O(log n).")
target = array[len(array) // 2]
start = time.perf_counter()
found = target in array                          # a linear scan; there is nothing better
t_scan = time.perf_counter() - start
print("   Finding an arbitrary element means scanning: %s, %.4f s" % (found, t_scan))
print("   Binary search would be WRONG here -- the array is not sorted, so")
print("   bisect would silently return nonsense. Try it and you get a plausible")
print("   answer that happens to be false.")
import bisect
i = bisect.bisect_left(array, target)
print("   bisect_left(array, %d) = %d, and array[%d] = %d  <- not the target"
      % (target, i, i, array[i] if i < len(array) else -1))

print()
print("2. The kth smallest is NOT at index k-1.")
ordered = sorted(array)
print("   %6s %14s %14s" % ("k", "array[k-1]", "true kth"))
for k in (1, 2, 3, 5, 10):
    print("   %6d %14s %14s"
          % (k, "{:,}".format(array[k - 1]), "{:,}".format(ordered[k - 1])))
print("   Only k = 1 is guaranteed. Even the SECOND smallest is merely known")
print("   to be one of the root's two children -- index 1 or index 2.")

print()
print("3. Iteration is not ordered.")
print("   first 8 of the array :", [int(x) for x in array[:8]])
print("   first 8 in order     :", [int(x) for x in ordered[:8]])

print()
print("4. Arbitrary deletion needs a search first, so it is Theta(n).")
print("   That is Java's PriorityQueue.remove(Object), measured at 82x slower")
print("   than poll() in section 1.5 -- and spelled almost identically.")
del big, array, ordered

A heap of 200,000 elements.

1. SEARCH is Theta(n), not O(log n).
   Finding an arbitrary element means scanning: True, 0.0036 s
   Binary search would be WRONG here -- the array is not sorted, so
   bisect would silently return nonsense. Try it and you get a plausible
   answer that happens to be false.
   bisect_left(array, 725581) = 99949, and array[99949] = 799865  <- not the target

2. The kth smallest is NOT at index k-1.
        k     array[k-1]       true kth
        1             20             20
        2             24             20
        3             20             24
        5             24             27
       10             35             48
   Only k = 1 is guaranteed. Even the SECOND smallest is merely known
   to be one of the root's two children -- index 1 or index 2.

3. Iteration is not ordered.
   first 8 of the array : [20, 24, 20, 36, 24, 27, 48, 59]
   first 8 in order     : [20, 20, 24, 24, 27, 35, 36, 37]

4. Arbitrary deletion needs a search first, so

**The four consequences, and the bug each one causes:**

| The heap does not promise | So this is wrong |
|---|---|
| a sorted array | binary-searching the array — it returns a plausible wrong answer |
| element $k$ at index $k-1$ | `heap[k-1]` for the $k$th smallest |
| ordered iteration | printing or iterating a `PriorityQueue` and expecting order |
| locating an element | `remove(x)` in a loop, which is $\Theta(n)$ each |

The **binary search** one is the most dangerous, because `bisect` on a heap array does not raise —
it returns an index, and the value there is simply wrong. Nothing fails; the answer is just false.

**Why the misconception survives:** at $n = 2$ a heap *is* always sorted, and at small $n$ it often
is by luck. Every hand-drawn example in a lecture is small. The distinction only becomes visible at
sizes where you have stopped checking by eye.

**The right mental model:** a heap is a **partial order**, and specifically it is the *weakest*
partial order that still puts the minimum where you can reach it. Everything the heap is good at
follows from that weakness — $\Theta(n)$ construction, $O(1)$ peek, a flat array with no pointers —
and everything it is bad at is the price. If you find yourself wanting a heap to be sorted, you
wanted a **sorted array** (static data) or a **balanced tree** (NB-08, dynamic data with ordered
queries), and you should use one of those instead.

***
# Part 4 - Tough questions

***

### Q1. State the heap property, and say what it does *not* guarantee.

<details><summary>Answer</summary>

> **Every node's key is $\le$ both its children's** (min-heap), and the tree is **complete** —
> every level full except possibly the last, filled left to right.

It is a **partial order**, and what it does not say is the important half: **nothing about
siblings**, nothing about cousins, nothing about any two nodes not on a common root-to-leaf path.

So the only located element is the **root**. The second smallest is one of the root's two children;
the seventh smallest could be almost anywhere. §3 measures the consequences — search, the $k$th
smallest, ordered iteration and arbitrary deletion are all $\Theta(n)$.

**The completeness half earns the whole structure**, because a complete tree needs no pointers:

$$\text{left}(i) = 2i+1,\qquad \text{right}(i) = 2i+2,\qquad \text{parent}(i)=\lfloor (i-1)/2 \rfloor$$

A heap of $n$ elements *is* an array of $n$ elements. That erases NB-04 §1.1's 48-bytes-per-node and
NB-04 §3.1's 30× pointer-chasing penalty at a stroke, and it is only possible because the invariant
is weak enough to maintain while keeping the tree complete — which a BST's invariant is not.

</details>

***

### Q2. Why is heapify $\Theta(n)$ and not $\Theta(n\log n)$?

<details><summary>Answer</summary>

**Because most nodes are near the bottom, where sifting down is cheap.**

Sift **down** from the last internal node backwards. A node at height $h$ costs $O(h)$, and there
are at most $n/2^{h+1}$ nodes at height $h$:

$$\sum_{h \ge 0} \frac{n\,h}{2^{h+1}} \;=\; \frac{n}{2}\sum_{h\ge0}\frac{h}{2^h} \;=\; \frac{n}{2}\cdot 2 \;=\; n$$

The whole argument is that $\sum h/2^h$ **converges** to 2. Half the nodes are leaves and cost
nothing; only the single root can cost $\log n$.

Building by $n$ pushes works the other way — it sifts **up**, and most nodes are at the bottom, so
most travel the full height.

**And here §1.3's measurement adds something the claim does not.** Heapify does at most **1.00
swaps per element** on every input tested. But repeated `push` is *also* linear on random input
(1.27/element) and on ascending input (0.00) — it only becomes $\Theta(n\log n)$ on **descending**
input, where it grows to 11.4, 14.7, then 18.0 swaps per element as $n$ grows tenfold.

So "heapify is $O(n)$, pushes are $O(n\log n)$" is only true for adversarial input. The honest
version: **heapify's cost is bounded regardless of input; repeated push's is a property of the
data.** That is this series' recurring question — who chooses the input — for the fourth time.

</details>

***

### Q3. Walk through push and pop.

<details><summary>Answer</summary>

**`push`:** append to the end of the array — which keeps the tree complete — then **sift up**:
while smaller than the parent, swap. Stops at the first parent that is smaller. $O(\log n)$.

**`pop`:** the root must go, but deleting it leaves a hole. So move the **last** element to the root
(keeping the tree complete) and **sift down**: while larger than its smallest child, swap with that
child. $O(\log n)$.

**`peek`:** $O(1)$ — it is `a[0]`.

**The detail people get wrong:** sift-down must compare against the **smaller** child. Swapping with
the larger one puts it above the smaller one and breaks the invariant on the other side. There is
no such choice in sift-up, which is why sift-down is the one with a bug in it.

**Why "move the last element to the root" is not arbitrary:** it is the only choice that keeps the
tree complete. Any other element would leave a hole in the middle, and the array layout depends on
there being no holes.

**The asymmetry worth noticing:** sift-up compares against **one** parent per level; sift-down
against **two** children. That is why §1.4's d-ary heaps make push cheaper and pop dearer, and why
the optimal $d$ is a trade rather than a maximum.

</details>

***

### Q4. Is a heap sorted?

<details><summary>Answer</summary>

**No, and this is the most common misconception about heaps.** §3 is entirely about it.

The invariant relates parents to children only. `[1, 20, 15, 38, 47, 24, ...]` is a perfectly valid
min-heap and is obviously not sorted.

**Four consequences, each a real bug:**

1. **Binary search on the array is wrong** — and it does not raise. `bisect` returns an index and
   the value there is simply not the target. §3 demonstrates it: a plausible answer that is false.
2. **The $k$th smallest is not at index $k-1$.** Only $k=1$ is guaranteed. Even the second smallest
   is merely known to be at index 1 or 2.
3. **Iteration is unordered.** Java's `PriorityQueue.toString()` prints heap order, and §1.5 shows
   it: `[1, 3, 2, 5, 9, 8]`.
4. **Arbitrary deletion needs a linear search**, which is `PriorityQueue.remove(Object)` — measured
   at **82× slower** than `poll` in §1.5, and spelled almost identically.

**Why the misconception survives:** §3 measured how often a heapified random array *happens* to be
sorted — **100% at $n=2$**, 52% at $n=3$, 34% at $n=4$, and 0% by $n=10$. At $n=2$ there is only one
parent-child pair, so "heap" and "sorted" coincide. Every hand-drawn lecture example is small, and
the distinction only becomes visible at sizes you stop checking by eye.

**Sorted output exists only if you drain the heap**, which is heapsort and costs $\Theta(n\log n)$.

</details>

***

### Q5. Find the k largest of n items.

<details><summary>Answer</summary>

**A min-heap of size $k$** — a *min*-heap, to find the *largest*. Its root is the smallest of the
best $k$ so far, which is exactly what to evict. Push each item; if the heap exceeds $k$, pop.

$\Theta(n\log k)$ time, **$\Theta(k)$ space**.

**The space bound is the point**, and §2.1 says so explicitly because the timing does not: a full
sort was competitive on time in memory (0.63 s against 0.12 s for two million items). But the heap
holds **10** elements against the sort's two million, so it runs on a stream that does not fit in
memory — where the sort cannot run at all.

**Alternatives and when they win:**

- **Sort and slice:** $\Theta(n\log n)$, needs all $n$ in memory, and is often fastest in practice
  for in-memory data because Timsort is very good. Perfectly reasonable when $n$ fits.
- **Quickselect** (NB-14): $\Theta(n)$ *expected* to partition around the $k$th, $\Theta(n^2)$ worst
  case. Fastest for in-memory data when you also want the elements *unordered*; needs the whole
  array and mutates it.
- **`heapq.nlargest(k, iterable)`** does exactly §2.1's algorithm, in C, and is what you should
  actually call.

**The trap:** building a max-heap of all $n$ and popping $k$ times is $\Theta(n + k\log n)$ — fine
asymptotically, and it needs all $n$ resident, which defeats the purpose.

</details>

***

### Q6. Maintain a running median.

<details><summary>Answer</summary>

**Two heaps facing each other** (§2.2):

- a **max**-heap for the lower half — its root is the largest small value;
- a **min**-heap for the upper half — its root is the smallest large value.

The roots straddle the median, so the median is $O(1)$; each insert is $O(\log n)$.

**The invariant, which is what to assert:** every element of the low heap $\le$ every element of the
high heap, and the sizes differ by at most one. After inserting into whichever side accepts the
value, rebalance by moving one root across if the sizes drift.

**Two practical notes:**

- **Python has no max-heap**, so the low half stores **negated** values. That is the standard trick
  and the standard source of sign errors — which is why §2.2's invariant checks the actual ordering
  rather than trusting the negation.
- **Decide what the median of an even count is** (the mean of the two middles, here) and be
  consistent. §2.2 verifies against a sorted-list reference at *every* step, not just the last.

**Where it generalises:** the same two-heap structure gives any fixed quantile by changing the size
ratio you rebalance to, and it is the standard way to compute streaming percentiles when exactness
matters. If approximate is acceptable, t-digest and similar sketches are far cheaper.

</details>

***

### Q7. Merge k sorted sequences.

<details><summary>Answer</summary>

**A heap of size $k$ holding one element from each source.** Pop the smallest, emit it, push the
next element from *that* source. $\Theta(N\log k)$ for $N$ elements, $\Theta(k)$ space.

**The implementation detail that matters:** push `(value, source_index)`, not `(value, iterator)`.
The index says where to refill from, **and it breaks ties without comparing the iterators**, which
are not orderable. Push a tuple whose second element is not comparable and Python raises the moment
two values tie — the same class of trap as NB-07 §1.4's `TreeSet` comparator.

**Why $\log k$ and not $\log N$:** the heap only ever holds one entry per source.

**Where this actually matters:** it is the merge step of **external sorting** — sort chunks that fit
in memory, write them out, then k-way merge the runs. That is how databases sort data larger than
RAM, and how LSM-tree engines compact. `heapq.merge` is this, and it is lazy, so it streams.

**And §2.3's honest measurement:** concatenating and sorting was *faster* in memory (0.09 s against
0.26 s for 400,000 elements), because Timsort detects the pre-sorted runs and merges them in C. The
heap's argument is space and streaming, not speed — the same shape of conclusion as Q5.

</details>

***

### Q8. Explain heapsort, and why nobody uses it.

<details><summary>Answer</summary>

**The algorithm:** heapify into a **max**-heap ($\Theta(n)$), then repeatedly swap the root with the
last element of the unsorted prefix and sift down over the shrunken range. The extracted maximum
lands exactly where it belongs, so the sorted region grows from the right, in place.

$\Theta(n\log n)$ **guaranteed**, $\Theta(1)$ extra space, **not stable**.

**On paper it dominates quicksort** — same average, better worst case, less space. §2.4 measures it
having no bad case at all: random, sorted and reverse-sorted inputs all give the same $n\log n$
ratios.

**In practice it is the slowest of the three, because of locality.** Quicksort scans contiguous
runs; heapsort's sift-down jumps to $2i+1$, then $4i+3$, then $8i+7$ — strides that double every
level and leave the cache almost immediately. It is the one place in this notebook where the flat
array is *not* enough: the layout is contiguous but the **access pattern** is not.

**Where it does earn its place:** **introsort** — the algorithm actually behind `std::sort` — runs
quicksort and switches to heapsort when the recursion depth suggests the quadratic case is
happening. Heapsort is the guaranteed-$O(n\log n)$ backstop that makes quicksort safe to deploy.
NB-13 covers this.

**And a use it is genuinely good for:** sorting in a hard memory bound, where $\Theta(1)$ extra
space is not negotiable and mergesort's $\Theta(n)$ buffer is unaffordable.

</details>

***

### Q9. Why is there no decrease-key, and what do you do instead?

<details><summary>Answer</summary>

**Because a heap cannot find an element.** To lower a key's priority you must locate it, and search
is $\Theta(n)$ (§3) — so `decreaseKey` would be $\Theta(n)$, which is useless to the algorithm that
wants it most, Dijkstra's (NB-21). Neither `heapq` nor `java.util.PriorityQueue` offers it.

**Two workarounds, and §2.5 measures both:**

1. **Indexed heap:** maintain a `key → array position` map, updated on **every swap**. Then
   `decrease_key` is $O(\log n)$. This is what the textbook Dijkstra analysis assumes, and keeping
   the map correct through every swap is NB-08 §2.2's augmentation discipline — which is why §2.5's
   invariant checks the map after every operation.
2. **Lazy deletion:** push a *new* entry with the better priority, leave the stale one, and skip
   stale entries when they surface.

**The measurement gives both sides a point.** Like-for-like in Python the indexed heap wins, because
its heap never grows past $n$ while lazy deletion's grows to about 2.7× — the algorithmic saving is
real. **And the C library's heap with lazy deletion beats both**, because the constant-factor gap is
larger than the saving. That is NB-04 §3 and NB-08 §2.4's lesson again: *your clever structure
competes with their fast simple one.*

**So: use lazy deletion with the library heap.** A dozen lines, no map to corrupt, and what most
production Dijkstra implementations and every event-simulation loop actually do. Reach for an
indexed heap when memory is tight or you need to *query* a current priority.

**And do not over-claim the asymptotics.** Lazy deletion pushes at most $E$ entries, so the heap is
$O(E)$ and $\log E = O(\log V)$ on simple graphs — Dijkstra is $O((V+E)\log V)$ either way. Lazy
deletion does not change the complexity class.

</details>

***

### Q10. When is a d-ary heap better than a binary one?

<details><summary>Answer</summary>

A **d-ary heap** gives each node $d$ children at $di+1 \ldots di+d$. The tree has $\log_d n$ levels,
so:

- **push gets cheaper** — sift-up compares one parent per level, over fewer levels: $O(\log_d n)$;
- **pop gets dearer** — sift-down finds the smallest of $d$ children per level: $O(d\log_d n)$,
  minimised near $d = e \approx 3$.

**By comparison count, $d=2$ and $d=4$ tie and everything bigger is worse** — §1.4 confirms the
measured counts track $d\log_d n$ closely (30.3, 31.6, 43.4, 67.2, 112.2 per element for
$d = 2,4,8,16,32$).

**By stopwatch, $d = 4$ wins**, doing *more* comparisons than $d=2$ in *less* time (1.61 s against
2.00 s). The reason is the array layout: half as many levels means half as many jumps to distant
parts of the array, and the four children compared at each step are **adjacent**, arriving together
in one cache line. It trades comparisons — cheap — for cache misses, which are not.

**So prefer $d = 4$ (or 8) when:**

- the workload is **push-heavy**, since push gets strictly cheaper as $d$ grows;
- the elements are small, so more children fit per cache line;
- specifically for **Dijkstra on dense graphs**, where decrease-key (a sift-up) dominates — the
  classic result is that $d = E/V$ is optimal there.

**Stay with $d=2$ when** the workload is pop-heavy, or when you want the simplest code and the
library's tuned implementation.

</details>

***

### Q11. What is a heap bad at?

<details><summary>Answer</summary>

Everything the invariant does not mention, which is nearly everything (§3):

- **Search: $\Theta(n)$.** No ordering to guide a descent. A hash table (NB-03) or a balanced tree
  (NB-08) does this in $O(1)$ or $O(\log n)$.
- **Arbitrary deletion: $\Theta(n)$**, because it needs the search first. Java's
  `PriorityQueue.remove(Object)` is measured at 82× slower than `poll` in §1.5.
- **Ordered iteration: $\Theta(n\log n)$** — you must drain it, destroying the heap, or copy first.
- **The $k$th smallest for $k > 1$:** not located. Only the root is.
- **Range queries, predecessor, successor:** all $\Theta(n)$. A balanced tree does them in
  $O(\log n)$.
- **Merging two heaps: $\Theta(n)$** — concatenate and re-heapify. This is the one a **binomial** or
  **Fibonacci** heap fixes, in $O(\log n)$ and $O(1)$ amortised respectively.

**The decision:**

| You need | Use |
|---|---|
| repeated access to the extreme | **heap** |
| lookup by key | hash table (NB-03) |
| ordered queries, ranges, successor | balanced tree (NB-08) |
| a sorted result, once | just sort |
| both extremes | two heaps, or a min-max heap |

**And the honest framing:** a heap is the *weakest* useful tree invariant. That weakness buys
$\Theta(n)$ construction, $O(1)$ peek and a pointer-free array; the price is that it answers exactly
one question.

</details>

***

### Q12. Why does a heap have good locality when NB-04 and NB-08 complained about trees?

<details><summary>Answer</summary>

**Because it is not made of nodes.** This is the notebook's structural punchline.

NB-04 §1.1 measured a Python linked-list node at 48 bytes against 8 for a list slot, and NB-04 §3.1
measured pointer-chasing traversal at ~30× slower than a contiguous array in Java. NB-08 §1.6 hit
the same wall from the other side: a binary tree of a billion keys is 30 random accesses per lookup,
which is why B-trees exist.

A heap escapes all of it because **the tree is implicit**. Completeness means the parent-child
relationships are *arithmetic on the index*, so there are no pointers, no per-node object headers,
no allocation per element, and the whole structure is one contiguous block that the prefetcher can
follow.

**The general lesson, worth more than the heap itself: whenever a tree is complete, it can live in
an array.** That is why heaps are used for priority queues, why segment trees (NB-12) are stored in
arrays the same way, and why Fenwick trees are barely recognisable as trees at all.

**And the honest caveat, because §2.4 found it:** contiguous *storage* is not the same as
contiguous *access*. Heapsort's sift-down jumps by $2i+1$, then $4i+3$, then $8i+7$ — strides that
double every level — so it leaves the cache almost immediately despite the perfect layout. That is
why heapsort loses to quicksort in practice, and it is a useful correction to "arrays are fast":
**the layout removes the pointer chase, and the access pattern still has to cooperate.** §1.4's
d-ary result is the same observation used constructively.

</details>

***

## Coding challenges

### Challenge 1 — a min-max heap

A heap gives you one extreme. Get both.

1. Implement a **min-max heap**: a single array where even levels satisfy the min property and odd
   levels the max property, so `find_min` and `find_max` are both $O(1)$ and both extractions are
   $O(\log n)$.
2. Verify against a sorted list over thousands of randomised operation sequences, with the
   alternating level invariant asserted after every operation. That invariant is the whole
   difficulty — sift-down must know which kind of level it is on and compare against
   *grandchildren* as well as children.
3. Measure it against two separate heaps with cross-references, and say which you would ship.

### Challenge 2 — measure the heapify constant properly

§1.3 measured heapify at 0.74 swaps per element on random input and 1.00 on descending. The proof
says "at most $n$". Close the gap.

1. Derive the *expected* number of swaps for random input, not just the upper bound.
2. Measure across many seeds at several sizes and report the mean with its spread, rather than one
   run per size as §1.3 does.
3. Find the input that actually maximises the swap count. Is descending order the worst, or is
   there something worse? Search for it and report what you find.
4. Repeat for d-ary heaps and see how the constant moves with $d$.

### Challenge 3 — Dijkstra three ways

§2.5 measured decrease-key against lazy deletion on a synthetic workload. Do it on the real one.

1. Implement Dijkstra (NB-21 previews it) three ways: indexed heap with `decrease_key`, lazy
   deletion with `heapq`, and a plain array with linear scan for the minimum.
2. Verify all three produce identical shortest-path distances on random graphs.
3. Measure them across a range of graph **densities**. The linear-scan version is $O(V^2)$ and
   should *win* on dense graphs — find the crossover.
4. Report which you would ship, and note that the answer depends on density, which is exactly the
   kind of thing an asymptotic comparison hides.

***
# Part 5 - Practice

| # | Exercise | The technique | Difficulty |
|---|---|---|---|
| 1 | A binary heap from scratch | Sift-up, sift-down, the array layout | ★★☆☆☆ |
| 2 | Kth largest in a stream | A size-k heap | ★★☆☆☆ |
| 3 | Merge k sorted lists | One heap, one element per source | ★★★☆☆ |
| 4 | Task scheduler / CPU intervals | A heap plus a cooldown queue | ★★★★☆ |
| 5 | Median from a data stream | Two heaps | ★★★☆☆ |
| 6 | Sliding window median | Two heaps **plus** deletion | ★★★★★ |
| 7 | Reorganise a string | Greedy by frequency, with a heap | ★★★☆☆ |
| 8 | Smallest range covering k lists | k-way merge with a window | ★★★★★ |

***

### 1. A binary heap from scratch

- **Brief:** `push`, `pop`, `peek`, and `heapify`. Then heapsort on top of it.
- **Good result:** differential-tested against `heapq` over thousands of randomised sequences with
  the heap property asserted after every operation (§1.2's harness).
- **The trap:** sift-down comparing against the **smaller** child, and the heapify start index. For
  a binary heap `n//2 - 1` happens to work; for d-ary it must be `(n-2)//d`, and §1.4 caught that
  exact bug with a differential test at $d=3$ after $d=2$ passed.

### 2. Kth largest in a stream

- **Brief:** a **min**-heap of size $k$; the root is the answer after each addition.
- **Good result:** $O(\log k)$ per element, $O(k)$ space, verified against a sorted list.
- **The trap:** using a max-heap because the question says "largest". Work out which root you need
  to *evict* and the direction follows. Also: what should it return before $k$ elements have
  arrived? Define it.

### 3. Merge k sorted lists

- **Brief:** §2.3 — push `(value, source_index)` so ties never compare the payload.
- **Good result:** $\Theta(N\log k)$, verified against concatenate-and-sort, handling empty lists
  and $k = 0$.
- **The trap:** pushing an object that is not comparable as the tie-breaker. It works until two
  values tie, then raises. Make the second tuple element an `int`, always.

### 4. Task scheduler

Tasks with a cooldown: the same task cannot run within $n$ ticks of itself. Minimise total time.

- **Brief:** a max-heap by remaining count, plus a **queue** holding tasks in cooldown with the tick
  they become available. Each tick, take the most frequent available task.
- **Good result:** verified against a brute-force simulation on small inputs.
- **The trap:** idle ticks. When the heap is empty but the cooldown queue is not, time still passes.
  There is also a closed-form answer for the minimum length — derive it and check your simulation
  against it, which is a much stronger test than examples.

### 5. Median from a data stream

- **Brief:** §2.2 — two heaps, rebalanced so the sizes differ by at most one.
- **Good result:** $O(\log n)$ per add, $O(1)$ per query, verified at every step.
- **The trap:** the negation for the max-heap, and rebalancing **after** every insert rather than
  before. Assert the cross-heap ordering; it catches sign errors immediately.

### 6. Sliding window median

The median of every window of size $k$ — §2.2 plus **removal**.

- **Brief:** two heaps with **lazy deletion** (§2.5): mark the departing element and purge it only
  when it reaches a root. Track each side's *effective* size separately from its raw length.
- **Good result:** $O(n\log k)$, verified against sorting each window.
- **The trap:** the size bookkeeping. Once heaps contain stale entries, `len(heap)` is no longer the
  count of live elements, so rebalancing must use the effective counts. This is the hardest problem
  in the notebook and the one that most rewards §2.5.

### 7. Reorganise a string

Rearrange characters so no two adjacent are equal, or report it is impossible.

- **Brief:** count frequencies, put them in a max-heap, and repeatedly take the **two** most
  frequent, emit both, and push back what remains.
- **Good result:** $O(n\log \Sigma)$; verified by checking no two adjacent characters match and the
  multiset is preserved.
- **The trap:** the impossibility condition — it is solvable iff the most frequent character appears
  at most $\lceil n/2 \rceil$ times. Derive that rather than discovering it, and check it up front.

### 8. Smallest range covering elements from k lists

Find the narrowest range containing at least one element from each of $k$ sorted lists.

- **Brief:** §2.3's k-way merge, but track the current **maximum** across the heap's contents as
  well. Each time you advance, the heap's root and that maximum bound a candidate range.
- **Good result:** $\Theta(N\log k)$, verified against brute force on small inputs.
- **The trap:** maintaining the maximum. It only ever increases as you advance sources, so it is a
  single variable, not another heap — realising that is the exercise. Stop as soon as any list is
  exhausted.

***
# Part 6 - Reading

## Start here

**1. *Introduction to Algorithms* (CLRS), chapter 6 — "Heapsort".**
> The reference treatment, and the source of §1.3's summation. Section 6.3 does the
> $\sum h/2^h = 2$ argument properly, and 6.5 covers priority queues including the
> `decreaseKey` that §1.5 explains real libraries do not offer. Short and complete.

**2. [CPython's `heapq` source](https://github.com/python/cpython/blob/main/Lib/heapq.py)** —
**Free.**
> Unusually well commented for a standard library module, and its header explains the array layout
> and the `heapreplace`/`heappushpop` distinction §2.1 uses. Worth reading for the note on why the
> module offers no `decrease_key`, which is §1.5's subject stated by the people who decided it.

**3. [Williams, *Algorithm 232: Heapsort*](https://dl.acm.org/doi/10.1145/512274.512284)** —
CACM 1964, and **Floyd, *Algorithm 245: Treesort 3*** the following year.
> Williams invented the heap; Floyd immediately improved the construction from $\Theta(n\log n)$ to
> $\Theta(n)$ — which is §1.3. Two one-page papers, a year apart, and the second is the one this
> notebook measures. A neat demonstration that the obvious construction is not the best one.

## The source behind each section

| Section | Where it comes from | Free? |
|---|---|---|
| 1.1 — the implicit array | **CLRS ch. 6.1** | 🔍 |
| 1.2 — sift-up / sift-down | **Williams**, CACM 1964; **CLRS ch. 6.2** | 🔍 |
| 1.3 — **heapify in $O(n)$** | **Floyd**, CACM 1964; **CLRS ch. 6.3** for the summation | 🔍 |
| 1.4 — d-ary heaps | **Johnson**, *Priority queues with update*, 1975; the cache argument is folklore, measured here | 🔍 |
| 1.5 — `PriorityQueue` | the JDK source and Javadoc | ✅ |
| 2.2 — two-heap median | folklore; no canonical source, which is why it is worth naming |— |
| 2.3 — k-way merge, external sorting | **Knuth, TAOCP vol. 3 §5.4** — external sorting in full | 🔍 |
| 2.4 — heapsort in practice | **Musser**, *Introspective Sorting and Selection Algorithms*, 1997 — introsort | 🔍 |
| 2.5 — decrease-key | **Fredman & Tarjan**, *Fibonacci heaps and their uses*, JACM 1987 | 🔍 |
| Q11 — mergeable heaps | **Vuillemin**, *A data structure for manipulating priority queues*, CACM 1978 (binomial heaps) | 🔍 |

**Legend:** ✅ free at the link · 🔍 search the exact title on
[Google Scholar](https://scholar.google.com)

### If you read only one

**Fredman & Tarjan on Fibonacci heaps**, and read it against §2.5.

The paper's whole motivation is the operation this notebook's heap cannot do: it achieves
`decrease_key` in $O(1)$ *amortised*, which improves Dijkstra from $O(E\log V)$ to
$O(E + V\log V)$ — a genuinely better bound, and one of the most cited results in the field.

Then notice that **essentially nobody uses Fibonacci heaps**. The constant factors are large, the
structure is intricate, and §2.5 measured why that matters: even a plain indexed binary heap lost to
lazy deletion on a fast library implementation. A better asymptotic bound lost to a simpler
structure with smaller constants — twice over.

That is the most useful pairing in this reading list. The paper is right, the bound is real, and the
engineering answer is still "use a binary heap with lazy deletion". Holding both of those at once is
the skill this series keeps trying to build.

Then read **CLRS chapter 6** for the proofs, and **Floyd's one-page 1964 note** for §1.3's
algorithm in its original form.

***
# Appendix

| Symptom | Cause | Fix |
|---|---|---|
| Binary search on the heap array returns nonsense | The array is **not sorted** (§3) | Drain the heap, or use a sorted array |
| `heap[k-1]` is not the $k$th smallest | Only the root is located (§3) | Pop $k$ times, or use quickselect (NB-14) |
| Iterating a `PriorityQueue` gives unordered output | Iteration is heap order (§1.5, §3) | Poll until empty, or sort a copy |
| `remove(x)` in a loop is quadratic | It is a $\Theta(n)$ linear search (§1.5) | Lazy deletion (§2.5) |
| Heap building is slow | $n$ pushes on descending input is $\Theta(n\log n)$ (§1.3) | Heapify — `heapq.heapify`, or `new PriorityQueue(collection)` |
| d-ary heap gives wrong output | Heapify start index: it is `(n-2)//d`, not `n//d - 1` (§1.4) | Caught only by a differential test at $d \ne 2$ |
| Sift-down corrupts the heap | Swapped with the **larger** child (§1.2) | Compare against the smaller one |
| `TypeError` comparing tuples in a k-way merge | Tie-break element is not orderable (§2.3) | Push `(value, source_index)` with an `int` index |
| Running median drifts wrong | Sign error in the negated max-heap (§2.2) | Assert the cross-heap ordering, not just the sizes |
| Indexed heap loses elements | Position map not updated on **every** swap (§2.5) | Update it inside `_swap`; check it in the invariant |
| Heapsort slower than quicksort | Sift-down strides double every level (§2.4) | Expected; use it as introsort's backstop |
| Every measured column reads 0.00 | `random.Random(n)` re-seeded inside a comprehension (§1.3) | Create the generator **once** |

## Checklist for heap code

- [ ] Is anything assuming the array is sorted (§3)?
- [ ] Is the $k$th smallest being read from index $k-1$ (§3)?
- [ ] Is `remove(x)` or a linear search being called in a loop (§1.5)?
- [ ] Is the heap built with **heapify** rather than $n$ pushes (§1.3)?
- [ ] For top-k, is the heap size $k$ — and is it a **min**-heap (§2.1)?
- [ ] In a k-way merge, is the tie-break element orderable (§2.3)?
- [ ] For a max-heap in Python, is every negation matched (§2.2)?
- [ ] If an indexed heap is used, is the position map updated in **every** swap and checked in the
      invariant (§2.5)?
- [ ] Would **lazy deletion with the library heap** be simpler and faster (§2.5)?
- [ ] Does the workload actually need a heap, or just one sort (§2.1, §2.3)?

## Where to go next

| Notebook | Why it follows |
|---|---|
| `sorting_comparison_zero_to_hero.ipynb` | Heapsort in context, and introsort — where §2.4 says heapsort earns its place |
| `sorting_linear_zero_to_hero.ipynb` | Quickselect, the other answer to §2.1's top-k |
| `graphs_paths_zero_to_hero.ipynb` | Dijkstra — the algorithm §2.5's decrease-key question exists for |
| [`balanced_trees_zero_to_hero.ipynb`](balanced_trees_zero_to_hero.ipynb) | The other way to guarantee $O(\log n)$: a strong invariant maintained, rather than a weak one exploited |

See [`README.md`](README.md) for the full roster and reading order.